In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

from hazrisk import risk as hzr

from phd_project.scripts import fixed_record_sets as fxr
import phd_project.scripts.metaregression_analysis as mra
import phd_project.scripts.fragility_data_models as fm
import phd_project.scripts.fragility_correction_analysis as fca
from phd_project.config import config

cfg = config.load_config()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# !!! Placeholder so I can continue to work out what models I want to run, while I am finishing the new MSAs
STUDY_NAME = "AvgSA_03__ks__vs__femap695"
STUDY = config.study(cfg, STUDY_NAME)

print(f"\nActive study: {config.study(cfg)['name']}\n")

# !!! Placeholder so I can continue to work out what models I want to run, while I am finishing the new MSAs
STUDY_NAME = "AvgSA_03__ks__vs__femap695"
STUDY = config.study(cfg, STUDY_NAME)
print(f"Temporary Study: {STUDY}")


FC_COMPARISON_PLOTS_DIR = config.study_dir(cfg, "proc_data", "fragility_statistical_analysis", STUDY["name"]) / "fc_comparison_plots"
FC_DATA_DIR = config.study_dir(cfg, "proc_data", "fragility_statistical_analysis", STUDY["name"]) / "fc_var_decomp_data"


Active study: AvgSA_03__ks__vs__femap695

Temporary Study: {'name': 'AvgSA_03__ks__vs__femap695', 'ss': 'AvgSA_03__ks', 'fx': 'femap695', 'im': 'AvgSA_03'}


In [3]:
IM_TAG = "AvgSA_03"
N_BOOT = 200
BOOTSTRAP_PTH = config.bootstrap_roots(cfg, STUDY["name"])
STUDY_PTH = BOOTSTRAP_PTH["study"]
theta_data = fm.load_fragility_data(BOOTSTRAP_PTH, IM_TAG, quantity="theta", verbose=False)
beta_data = fm.load_fragility_data(BOOTSTRAP_PTH, IM_TAG, quantity="beta", verbose=False)
joint = fm.JointModelRegistry(
    theta_data, beta_data,
    cache_dir=BOOTSTRAP_PTH["study"] / "models",
)
joint.fit("A8f_bc_hb", n_boot=N_BOOT, verbose=True)
joint.cache_summary()
model = joint["A8f_bc_hb"]

[cache] A8f_bc_hb [REML]: loaded (200 saved, using 200)
cache: 1 loaded, 0 fitted


In [4]:
model.coef_table()

estimate  se_model   se_boot     ci_lo     ci_hi
outcome coefficient                                                            
theta   intercept              0.012005  0.015528  0.015502 -0.019841  0.044767
        d_ln_ds595_mean_unif  -0.075289  0.006840  0.007093 -0.089012 -0.061795
        d_ln_ds595_sigma_unif -0.002298  0.006773  0.007067 -0.013891  0.013301
        ln_T1                 -0.005239  0.012518  0.012303 -0.029785  0.018400
        ln_overstrength        0.013825  0.010000  0.010589 -0.007710  0.031294
        ln_ductility          -0.020005  0.008539  0.008769 -0.036913 -0.004379
beta    intercept              0.001729  0.044421  0.047277 -0.100657  0.105788
        d_ln_ds595_sigma_unif  0.076643  0.017725  0.019437  0.033235  0.112694

The models were fitted on normalised and standardised data. We need to present the relationships in real units. So lets convert them back.

In [5]:
# Coefficients on the standardised scale, one per (outcome, name) column of X.
coef_std = pd.Series(
    model.beta,
    index=pd.MultiIndex.from_tuples(model.x_names, names=["outcome", "coefficient"]),
    name="estimate_std",
)
print("Standardised Coefficients:")
print(coef_std)

# get the scaling -- mean and standard deviation of the covariates.
scaling = pd.DataFrame(
    [(q, cov, sc["mean"], sc["sd"])
     for q, covs in model.scaling.items() for cov, sc in covs.items()],
    columns=["outcome", "covariate", "mean", "sd"],
).set_index(["outcome", "covariate"])
print("\nScaling factors:")
print(scaling)


Standardised Coefficients:
outcome  coefficient          
theta    intercept                0.012005
         d_ln_ds595_mean_unif    -0.075289
         d_ln_ds595_sigma_unif   -0.002298
         ln_T1                   -0.005239
         ln_overstrength          0.013825
         ln_ductility            -0.020005
beta     intercept                0.001729
         d_ln_ds595_sigma_unif    0.076643
Name: estimate_std, dtype: float64

Scaling factors:
                                   mean        sd
outcome covariate                                
theta   d_ln_ds595_mean_unif  -0.040804  0.595479
        d_ln_ds595_sigma_unif  0.012986  0.215407
        ln_T1                 -0.511829  0.309219
        ln_overstrength        1.110787  0.301043
        ln_ductility           1.289465  0.455614
beta    d_ln_ds595_sigma_unif  0.012986  0.215407


In [6]:
# Denormalise the coefficients to get them in natural units
def denormalise_coefficients(coefs: pd.Series, scaling: dict, outcome: str) -> pd.Series:
    """Convert one outcome's coefficients from standardised to natural units.
       outcome: either "theta" or "beta"

        slope:      b_j = gamma_j / s_j
        intercept:  b_0 = m - sum_j gamma_j * mu_j / s_j

    `coefs` is indexed by coefficient name ("intercept", then covariates) for
    a single outcome. It works equally on the point estimate and on one
    bootstrap replicate, because the transform is linear.
    """
    sc = scaling[outcome]
    covs = [c for c in coefs.index if c != "intercept"]
    out = {}
    # Slopes: dividing by the SD turns "per SD of x" into "per unit of x".
    for c in covs:
        out[c] = coefs[c] / sc[c]["sd"]
    # Intercept: m is the fit AT THE MEANS of all the covariates. Moving it to 
    # x = 0 means subtracting each slope times the distance from 0 to the mean.
    out["intercept"] = coefs["intercept"] - sum(out[c] * sc[c]["mean"] for c in covs)
    return pd.Series(out)[["intercept"] + covs]

# convert the coefficients of the model to real units
natural = {q: denormalise_coefficients(coef_std.xs(q, level="outcome"), model.scaling, q)
           for q in ("theta", "beta")}


We now have three forms of the equation:
1. In log units: $$\ln{\frac{\theta_{ss}}{\theta_{fx}}}=b_0+\sum_j{b_j\ln{X_j}}$$
2. In normal units: $$\frac{\theta_{ss}}{\theta_{fx}}=\exp{b_0}\prod_j{X_j^{b_j}}$$

Writing (2) out in full for model A8f_bc_hb would be: !!TODO!! Update the sigma part of the equations$$\frac{\theta_{ss}}{\theta_{fx}}=e^{0.003075}\left(\frac{\mu_{ln(dur)}^{ss}}{\mu_{ln(dur)}^{fx}}\right)^{-0.125830}\left(\frac{\sigma_{ln(dur)}^{ss}}{\sigma_{ln(dur)}^{fx}}\right)^{-0.022404}T_1^{-0.017343}\Omega^{0.045896}\mu_D^{-0.043496}$$

$$\frac{\beta_{ss}}{\beta_{fx}}=e^{-0.009417}\left(\frac{\sigma_{ln(dur)}^{ss}}{\sigma_{ln(dur)}^{fx}}\right)^{0.572369}$$

In [7]:
natural["theta"]

intercept                0.003918
d_ln_ds595_mean_unif    -0.126434
d_ln_ds595_sigma_unif   -0.010669
ln_T1                   -0.016943
ln_overstrength          0.045925
ln_ductility            -0.043908
dtype: float64

In [8]:
natural["beta"]

intercept               -0.002892
d_ln_ds595_sigma_unif    0.355807
dtype: float64

In [9]:
model.T_total

array([[0.00226716, 0.00083611],
       [0.00083611, 0.00918573]])

In [10]:
model.rho_tot

0.18321658612673408

In [11]:
data = pd.read_csv(r"C:\Users\clemettn\Documents\phd\data_processed\10_fragility_curve_statistical_analysis\studies\AvgSA_03__ks__vs__femap695\fragility_regression_dataset_AvgSA_03.csv")

In [12]:
dir(model.contrast.theta)

['V',
 '__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'corrected',
 'diag_V',
 'forest_frame',
 'g',
 'k',
 'label',
 'labels',
 'n',
 'name',
 'quantiles',
 'quantity',
 'rows',
 'scope',
 'site_codes',
 'summary',
 'terms',
 'v',
 'y',
 'y_hat']

In [13]:
SITE = 33
N_STOREYS = 5

In [14]:
ida_data = data[["site", "n_storeys", "ida_avg_theta", "ida_avg_beta", "ida_avg_beta_bc", "ida_avg_var_log_theta", "ida_avg_var_log_beta", "ida_avg_cov_log_theta_log_beta"]]
ida_data.set_index(["site", "n_storeys"], inplace=True, drop=True)

site_ida_data = ida_data.loc[[(SITE, N_STOREYS)]]
site_ida_data

true_msa_data = data[["site", "n_storeys", "msa_ss_theta", "msa_ss_beta"]]
true_msa_data.set_index(["site", "n_storeys"], inplace=True, drop=True)
true_msa_data = true_msa_data.loc[[(SITE, N_STOREYS)]]

In [15]:
## Set up for site 33 / 5s
str_ps = {
    "T1": 0.748,
    "Omega": 2.645,
    "mu_T": 3.367,
}
fx_ps = {
    "theta_fx": 0.4684,
    "beta_fx": 0.1854,          # bias corrected value if its available
    "V_sq_theta": 0.00144,
    "V_sq_beta": 0.02050,
    "V_theta_beta": 0.00134,
    "rho_V": 0.24663,
    "RSD595_gm": 12.12,
    "sigma_lnRSD595": 0.55,
}
site_ps = {
    "RSD595_gm": 9.059,
    "sigma_lnRSD595": 0.528,
}

model_coeffs = {
    "intercept": [0.00392, -0.00289],
    "ln(mu_RSD595_ratio)": [-0.12643, 0],
    "ln(sigma_lnRSD595_ratio)": [-0.01067, 0.3558],
    "ln(T1)": [-0.01694, 0],
    "ln(Omega)": [0.04593, 0],
    "ln(mu_T)": [-0.04391, 0],
}

model_variance = {
    "T_sq_theta": 0.00227,
    "T_sq_beta": 0.00919,
    "T_theta_beta": 0.00084,
    "rho_T": 0.1832,
}

# Processing
model_coeffs = pd.DataFrame.from_dict(model_coeffs, orient="index", columns=["theta_coeffs", "beta_coeffs"])
covariates = {
    "intercept": 1,
    "ln(mu_RSD595_ratio)": np.log(site_ps["RSD595_gm"] / fx_ps["RSD595_gm"]),
    "ln(sigma_lnRSD595_ratio)": np.log(site_ps["sigma_lnRSD595"] / fx_ps["sigma_lnRSD595"]),
    "ln(T1)": np.log(str_ps["T1"]),
    "ln(Omega)": np.log(str_ps["Omega"]),
    "ln(mu_T)": np.log(str_ps["mu_T"]),
}
covariates = pd.Series(covariates)

In [16]:
site_structure_combis = ida_data.index.to_list()

HAZARD_SIGMA = 4        # PSHA epsilon truncation level of the hazard curve
HAZARD_STAT = "mean"    # logic-tree statistic
with open(cfg["proc_data"][f"{IM_TAG}_hazard_curves_{HAZARD_SIGMA}sig"], "rb") as f:
    _hcs = pickle.load(f)

results = {}
for site, n_storeys in site_structure_combis:
    # site hazard
    site_hc = np.asarray(_hcs[site]["AvgSA"][HAZARD_STAT], dtype=float)   # (n_iml, 2)
    im_hc = site_hc[:, 0]
    H_hc = site_hc[:, 1]

    # fragility curve and mafe assessment
    results[(site, n_storeys)] = fca.fragility_variance_assessment(
            joint, model, site, n_storeys, im_hc, H_hc, 0.05, iml_params=[0.01, 3.0, 1000],
        plot=True, fc_fig_path=FC_DATA_DIR, var_fig_path=FC_DATA_DIR, plot_show=False, dpi=300)

with open(FC_DATA_DIR/"fragility_assessment_data.pickle", "wb") as file:
    pickle.dump(results, file)
    # # fragility curve comparisons
    # fc_out, fig, ax = fca.fragility_comparison(joint, model, site, n_storeys, 0.05, plot=True, plot_show=False, fig_path=FC_COMPARISON_PLOTS_DIR)
    # fc_comparison_results[(site, n_storeys)] = (fc_out, fig, ax)

    # # variance decompositions
    

    # var_decomp_out = fca.mafe_variance_decomposition(fc_out["theta_msa_est"], fc_out["beta_msa_est"], im_hc, H_hc, joint, model, site, n_storeys)
    

    # contributions = list(var_decomp_out["var_source_decomp_normalised"].values())
    # tags = [fca.VAR_CONTRIBUTION_TAG_MAP[k] for k in var_decomp_out["var_source_decomp_normalised"].keys()]
    # fig, ax = fca.plot_variance_contributions(contributions, tags, fp=FC_DATA_DIR/ f"site_{site}_{n_storeys}s_var_decomp.png")

    # var_decomp_results[(site, n_storeys)] = (var_decomp_out, fig, ax)



## Assessment of the correction across all site/structure rows

The plots below summarise the per-row results of `fragility_variance_assessment` across all 120 site/structure rows:

- **Variance decomposition:** for each of the 12 sources, its % share of $\mathrm{Var}[\ln\lambda_C]$ and its absolute contribution; the 3 FOSM terms of the total; and a stacked share by source ($V$, $T$, $F$, $C$).
- **Accuracy:** $\ln(\lambda/\lambda_{MSA})$ for IDA-FX and for the corrected estimate, and the improvement
  $$\Delta=|\ln(\lambda_{IDA}/\lambda_{MSA})|-|\ln(\lambda_{est}/\lambda_{MSA})|$$
  ($+$ = the estimate is closer to MSA-SS). The right-hand axis shows $e^{\Delta}=EF_{IDA}/EF_{est}$, the ratio of the two error factors. $\Delta$ is plotted three ways: coloured by sign, coloured by the duration covariate, and grouped by design group.
- **Calibration:** a parity plot, and standardised errors for the corrected estimate and for IDA-FX
  $$z=\frac{\ln(\lambda/\lambda_{MSA})}{\sqrt{\mathrm{Var}[\ln\lambda]+\mathrm{Var}[\ln\lambda_{MSA}]}}$$
  compared with $N(0,1)$.
- **MAFC uncertainty:** the MAFC of every row with its 95% confidence interval, one figure per storey group, and a table of whether the IDA-FX and estimated MAFC lie inside the MSA-SS interval.

All figures are saved in `FC_DATA_DIR`.

In [ ]:
# Reload the saved per-row results and the hazard curves, so this section can be
# run without re-running the (slow) assessment loop above. HAZARD_SIGMA and
# HAZARD_STAT must match the values the loop used to produce the results.
HAZARD_SIGMA = 4
HAZARD_STAT = "mean"
with open(FC_DATA_DIR / "fragility_assessment_data.pickle", "rb") as f:
    results = pickle.load(f)
with open(cfg["proc_data"][f"{IM_TAG}_hazard_curves_{HAZARD_SIGMA}sig"], "rb") as f:
    _hcs = pickle.load(f)

# One row per (site, n_storeys), ordered 3-storey then 5-storey, each in site order.
# Holds the 12 variance contributions (absolute and %), the total-variance terms,
# the three MAFCs, their log errors, the improvement Delta, the fragility
# parameters, and the design group and duration covariate taken from the
# regression dataset `data`. The hazard curves are used in the next cell.
assessment_df = fca.assessment_frame(results, data)

# Sanity checks: the three total terms and the 12 sources both sum to Var[ln MAFC],
# so the % shares of each row sum to 100.
assert np.allclose(assessment_df[[f"tot_{t}" for t in fca.VAR_TERMS]].sum(axis=1), assessment_df["var_tot"])
assert np.allclose(assessment_df.filter(like="pct_").sum(axis=1), 100)
print(f"{len(assessment_df)} rows")
assessment_df[["mafe_true", "mafe_ida", "mafe_est", "ln_ratio_ida", "ln_ratio_est",
               "improvement", "var_tot"]].describe()

### Uncertainty in the MAFC

The sampling uncertainty of the MSA-SS and IDA-FX MAFC is found with the first-order second-moment (FOSM) rule (Ang & Tang 2007, eq. 4.51, p. 186):
$$\mathrm{Var}[\ln\lambda]\approx s_\theta^2\,\sigma^2_{\ln\theta}+s_\beta^2\,\sigma^2_{\ln\beta}+2\,s_\theta s_\beta\,\sigma_{\ln\theta\ln\beta},\qquad s_x=\frac{\partial\ln\lambda}{\partial\ln x}$$
The slopes are evaluated at each method's own $(\theta,\beta)$ on the site's own hazard curve. The parameter covariance is taken from the same bootstrap replicates the regression model uses (the first $r$ replicates). The corrected estimate uses its full predictive variance from the decomposition above.

The 95% interval of each MAFC is $\lambda\,e^{\pm1.96\sqrt{\mathrm{Var}[\ln\lambda]}}$. For IDA-FX and the estimate, $z=\ln(\lambda/\lambda_{MSA})/\sqrt{\mathrm{Var}[\ln\lambda_{MSA}]}$ measures the distance from the MSA-SS value in units of MSA-SS's own uncertainty, so "inside the MSA-SS 95% CI" is the same as $|z|\le1.96$.

In [ ]:
# FOSM sampling variance of ln MAFC for MSA-SS (true) and IDA-FX, row by row.
msa_tag, ida_tag = model.contrast.terms
var_ln_mafe = {"true": [], "ida": []}
for site, n_storeys in assessment_df.index:
    hc = np.asarray(_hcs[site]["AvgSA"][HAZARD_STAT], dtype=float)   # (n_iml, 2): IML, rate
    row = assessment_df.loc[(site, n_storeys)]
    for key, tag, theta, beta in [("true", msa_tag, row["theta_msa_true"], row["beta_msa_true"]),
                                  ("ida", ida_tag, row["theta_ida"], row["beta_ida"])]:
        # 2x2 covariance of (ln theta, ln beta) over the model's bootstrap replicates
        cov = fca.parameter_sampling_covariance(joint, model, site, n_storeys, tag)
        # combine with the slopes d ln(lambda)/d ln(theta), d ln(lambda)/d ln(beta)
        var_ln_mafe[key].append(fca.ln_mafe_sampling_variance(
            theta, beta, cov[0, 0], cov[1, 1], cov[0, 1], hc[:, 0], hc[:, 1]))
assessment_df["var_ln_mafe_true"] = var_ln_mafe["true"]
assessment_df["var_ln_mafe_ida"] = var_ln_mafe["ida"]

# The IDA variance is the "sampling" term of the decomposition propagated at the
# IDA parameters, so both must come from the same replicates.
assert np.allclose(
    [results[k][1]["fragility_variance"].loc["sampling", "theta"] for k in assessment_df.index],
    [fca.parameter_sampling_covariance(joint, model, *k, ida_tag)[0, 0] for k in assessment_df.index])

# 95% confidence intervals of all three MAFCs, the z-values of IDA-FX and the
# estimate against MSA-SS, and whether each lies inside the MSA-SS interval.
assessment_df = fca.mafe_confidence_intervals(assessment_df, level=0.95)

# Save every variance, interval and z-value, one row per site/structure.
uncertainty_cols = [
    "mafe_true", "var_ln_mafe_true", "ci_lo_true", "ci_hi_true",
    "mafe_ida", "var_ln_mafe_ida", "ci_lo_ida", "ci_hi_ida",
    "mafe_est", "var_ln_mafe_est", "ci_lo_est", "ci_hi_est",
    "ida_in_ci", "z_ida", "z_ida_combined",
    "est_in_ci", "z_est", "z_est_combined",
]
assessment_df[uncertainty_cols].to_csv(FC_DATA_DIR / "mafe_uncertainty.csv")

# Per-row summary: is each estimate inside the MSA-SS 95% CI, and its z-value.
mafe_ci_table = assessment_df[["mafe_true", "ci_lo_true", "ci_hi_true",
                               "mafe_ida", "ida_in_ci", "z_ida",
                               "mafe_est", "est_in_ci", "z_est"]]
print("Rows inside the MSA-SS 95% CI:")
print(fca.mafe_ci_summary(assessment_df))
mafe_ci_table

### Local hazard-curve slope $k$ at the fragility median

> **Provisional.** These results are on the placeholder study `AvgSA_03__ks__vs__femap695`, not the main-line record set.

**Purpose.** If the hazard curve is locally a power law, $H(x) = k_0\,x^{-k}$, the collapse risk has the closed form (Cornell et al. 2002, J. Struct. Eng. 128(4))

$$\lambda_C \approx H(\theta)\,\exp\!\left(\tfrac12 k^2\beta^2\right),$$

so the sensitivity of $\ln\lambda_C$ to the two fragility parameters is

$$\frac{\partial\ln\lambda_C}{\partial\ln\theta} = -k, \qquad \frac{\partial\ln\lambda_C}{\partial\ln\beta} = k^2\beta^2 .$$

The slope $k$ is therefore what converts a difference in $\theta$ (or $\beta$) between two arms into a difference in MAFC. This section finds $k$ for every row and every arm, and checks how well the power-law closed form describes the numerical MAFCs computed above.

**Definition.** For row $i$ and arm $X \in \{\text{IDA-FX}, \text{MSA-SS}, \text{MSA-SS}_{est}\}$ the local hazard slope is the negative log-log slope of row $i$'s site hazard curve at that arm's median:

$$k_{X,i} = -\left.\frac{d\,\ln H_{\sigma[i]}(x)}{d\,\ln x}\right|_{x=\theta_{X,i}}$$

$k$ is positive for a decreasing hazard curve. Both storeys at a site share one hazard curve (IM `AvgSA_03`, the IM of every fragility here).

**The hazard curve.** $\ln H$ is interpolated against $\ln x$ by PCHIP, exactly as `hzr.numerical_mafe_pchip` does when it computes the MAFCs, so the slopes belong to the same curve the risk integral used. Trailing zero rates are dropped and the curve is not extrapolated: a slope that needs $H$ outside the positive part of the curve is NaN.

**Three estimates**, each per arm:

- `k_local` — the point slope at $\theta$, by central difference with step $\delta = 0.01$ in $\ln x$. Describes the curve at the median:
$$k_{local} = -\frac{\ln H(\theta e^{\delta}) - \ln H(\theta e^{-\delta})}{2\delta}$$
- `k_window` — the secant slope over $\theta e^{\pm\beta}$, the range that contributes most of the risk integral. Smooths over local wiggles:
$$k_{window} = -\frac{\ln H(\theta e^{\beta}) - \ln H(\theta e^{-\beta})}{2\beta}$$
- `k_fosm` — the slope the risk integral actually responds to, $-\partial\ln\lambda_C/\partial\ln\theta$ from the numerical MAFC (`fca.get_fosm_mafe_slopes`, $\delta = 0.01$). Under a pure power law it equals $k$; the difference from `k_local` is the effect of curvature in the hazard curve:
$$k_{fosm} = -\frac{\ln\lambda_C(\theta e^{\delta},\beta) - \ln\lambda_C(\theta e^{-\delta},\beta)}{2\delta}$$

Also stored per arm: `ln_H` $=\ln H(\theta)$; `kbeta2` $= k_{local}^2\beta^2$, the closed-form prediction of $\partial\ln\lambda_C/\partial\ln\beta$; and `s_beta_fosm`, the same derivative taken numerically from the MAFC, to compare with it.

In [ ]:
# Helpers for the local hazard slope k. Small and notebook-local on purpose:
# fca is not changed for this diagnostic.
from hazrisk import hazard as haz

K_DELTA = 0.01                         # step in ln x for central differences; the same step
                                       # fca.get_fosm_mafe_slopes uses, so k_local and k_fosm
                                       # are differenced on the same scale
K_ARMS = ("ida", "msa_true", "msa_est")  # column suffixes, matching assessment_df


def ln_hazard_interpolator(ims, rates):
    """PCHIP interpolant of ln H against ln x for one site's hazard curve.

    Uses hazrisk's own builder (hazard._pchip_log_log), the exact object behind
    hzr.numerical_mafe_pchip, so every slope below is a slope of the same curve the
    MAFCs were integrated over. It is a private helper; calling it rather than
    copying it guarantees the two cannot drift apart. The builder drops the
    trailing zero rates and does not extrapolate: f(ln x) is NaN outside the
    positive part of the curve, which is what makes the NaN guards below work.

    Returns f, with f(ln x) = ln H(x) and f(ln x, 1) = d ln H / d ln x.
    """
    return haz._pchip_log_log(ims, rates)


def nan_reason(ln_pts, ims, f):
    """Why a slope that needed ln H at the points ln_pts came out NaN.

    'outside IML range': a point lies beyond the tabulated IMLs of the curve.
    'H = 0 there':       a point lies inside the IML range but above the last
                         positive rate, where ln H does not exist.
    Anything else is reported as a non-finite result.
    """
    ln_pts = np.atleast_1d(ln_pts)
    if np.any(ln_pts < np.log(ims[0])) or np.any(ln_pts > np.log(ims[-1])):
        return "outside IML range"
    if np.any(ln_pts > f.x[-1]):       # f.x are the knots of the trimmed (H > 0) curve
        return "H = 0 there"
    return "non-finite result"


def point_slope(f, theta, delta=K_DELTA):
    """k_local: -d ln H / d ln x at theta, by central difference with step delta in ln x.
    NaN if theta*exp(+-delta) is off the positive part of the curve."""
    x = np.log(theta)
    return float(-(f(x + delta) - f(x - delta)) / (2 * delta))


def window_slope(f, theta, beta):
    """k_window: secant slope of ln H between theta*exp(-beta) and theta*exp(+beta),
    i.e. averaged over the band that carries most of the risk integral."""
    x = np.log(theta)
    return float(-(f(x + beta) - f(x - beta)) / (2 * beta))


def fosm_slopes(theta, beta, ims, rates, f, delta=K_DELTA):
    """(k_fosm, s_beta_fosm) from the numerical MAFC.

    fca.get_fosm_mafe_slopes returns d ln(lambda)/d ln(theta) and
    d ln(lambda)/d ln(beta) by central differences of hzr.numerical_mafe_pchip.
    k_fosm is minus the first; the second is the numerical counterpart of the
    closed-form k^2 beta^2. The full (ims, rates) are passed, exactly as the
    assessment loop passed them, so these are the derivatives of the MAFCs above.

    The function has no range check of its own, so theta*exp(+-delta) is checked
    here against the positive part of the curve: NaN rather than a silent
    extrapolation.
    """
    x = np.log(theta)
    if x - delta < f.x[0] or x + delta > f.x[-1]:
        return np.nan, np.nan
    s_theta, s_beta = fca.get_fosm_mafe_slopes(theta, beta, delta, ims, rates)
    k = -s_theta if np.isfinite(s_theta) else np.nan
    s_beta = s_beta if np.isfinite(s_beta) else np.nan
    return float(k), float(s_beta)


def grid_spacing_at(f, theta):
    """Width in ln IML of the hazard-curve grid interval that contains theta.

    Diagnostic for the choice of delta: it shows how fine the step is relative
    to the tabulated curve. NaN if theta is off the positive part of the curve.
    """
    x = np.log(theta)
    if x < f.x[0] or x > f.x[-1]:
        return np.nan
    j = min(np.searchsorted(f.x, x, side="right") - 1, len(f.x) - 2)
    return float(f.x[j + 1] - f.x[j])

In [ ]:
# Local hazard slope k at theta for every row and every arm.
# One row per (site, n_storeys), in the order of assessment_df. The hazard curve is
# the one the assessment loop used: _hcs[site]["AvgSA"][HAZARD_STAT], column 0 the
# IML and column 1 the annual rate. It is keyed by site only, so both storeys at a
# site share it, and its interpolant is built once per site.
k_interp = {}     # site -> PCHIP interpolant of ln H against ln x
k_records = []    # one dict per row, turned into the DataFrame below
k_nan_log = []    # one entry per NaN value: where it is and why

for site, n_storeys in assessment_df.index:
    hc = np.asarray(_hcs[site]["AvgSA"][HAZARD_STAT], dtype=float)   # (n_iml, 2): IML, rate
    ims, rates = hc[:, 0], hc[:, 1]
    if site not in k_interp:
        k_interp[site] = ln_hazard_interpolator(ims, rates)
    f = k_interp[site]

    # theta and beta of each arm. IDA-FX and MSA-SS come from assessment_df; the
    # corrected MSA-SS estimate is not a column there, so it is taken from the
    # fragility_comparison output stored in results (the dict assessment_frame
    # also reads). results is in loop order, so it is looked up by key.
    row = assessment_df.loc[(site, n_storeys)]
    fc_out = results[(site, n_storeys)][0]
    arm_params = {
        "ida": (row["theta_ida"], row["beta_ida"]),
        "msa_true": (row["theta_msa_true"], row["beta_msa_true"]),
        "msa_est": (fc_out["theta_msa_est"], fc_out["beta_msa_est"]),
    }

    rec = {"site": site, "n_storeys": n_storeys, "design_group_id": row["design_group_id"]}
    for arm, (theta, beta) in arm_params.items():
        x = np.log(theta)
        ln_H = float(f(x))
        k_local = point_slope(f, theta)
        k_window = window_slope(f, theta, beta)
        k_fosm, s_beta_fosm = fosm_slopes(theta, beta, ims, rates, f)
        rec.update({
            f"theta_{arm}": theta,
            f"beta_{arm}": beta,
            f"ln_H_{arm}": ln_H,
            f"k_local_{arm}": k_local,
            f"k_window_{arm}": k_window,
            f"k_fosm_{arm}": k_fosm,
            f"kbeta2_{arm}": k_local**2 * beta**2,          # closed-form d ln(lambda)/d ln(beta)
            f"s_beta_fosm_{arm}": s_beta_fosm,              # numerical d ln(lambda)/d ln(beta)
        })
        # Record every NaN with the ln x points the value needed, so the reason
        # (outside the IML range, or H = 0 there) can be reported.
        needed = {
            "ln_H": [x],
            "k_local": [x - K_DELTA, x + K_DELTA],
            "k_window": [x - beta, x + beta],
            "k_fosm": [x - K_DELTA, x + K_DELTA],
            "s_beta_fosm": [x - K_DELTA, x + K_DELTA],
        }
        for col, pts in needed.items():
            if not np.isfinite(rec[f"{col}_{arm}"]):
                k_nan_log.append({"site": site, "n_storeys": n_storeys, "arm": arm,
                                  "column": f"{col}_{arm}", "reason": nan_reason(pts, ims, f)})

    rec["dln_ims_at_theta_ida"] = grid_spacing_at(f, arm_params["ida"][0])
    k_records.append(rec)

k_df = pd.DataFrame(k_records).set_index(["site", "n_storeys"])

# Checks: 120 rows in assessment_df's order, and the IDA / MSA-SS parameters are
# exactly the ones the MAFCs were computed with.
assert len(k_df) == 120, f"expected 120 rows, got {len(k_df)}"
assert k_df.index.equals(assessment_df.index)
for arm in ("ida", "msa_true"):
    for p in ("theta", "beta"):
        assert np.array_equal(k_df[f"{p}_{arm}"].to_numpy(), assessment_df[f"{p}_{arm}"].to_numpy())

# NaN report: every NaN value, where it is and why. Nothing is extrapolated.
if k_nan_log:
    print(f"{len(k_nan_log)} NaN values:")
    print(pd.DataFrame(k_nan_log).to_string(index=False))
else:
    print("No NaN values: every evaluation point lies on the positive part of its hazard curve.")

# k must be positive wherever it is finite (the hazard curve decreases). All
# failures are collected and printed before the assert, not just the first one.
k_cols = [c for c in k_df.columns if c.startswith(("k_local_", "k_window_", "k_fosm_"))]
k_fail = k_df[k_cols].stack().loc[lambda s: s <= 0]
if len(k_fail):
    print(f"{len(k_fail)} non-positive k values:")
    print(k_fail.to_string())
assert k_fail.empty, "k <= 0 found; see the list above"

k_df.to_csv(FC_DATA_DIR / "hazard_slope_k_at_theta.csv")
print(f"Saved {len(k_df)} rows to {FC_DATA_DIR / 'hazard_slope_k_at_theta.csv'}")

In [ ]:
# Sensitivity of k_local_ida to the difference step delta.
# The hazard curve is tabulated on a grid in ln IML (spacing printed below). With a
# piecewise-linear interpolant a delta smaller than that spacing would return the
# slope of one straight segment, which jumps at every knot. The interpolant here is
# PCHIP, whose slope is continuous, so the central difference should converge to
# the exact derivative f(ln theta, 1) as delta shrinks. This cell checks that.
print("Grid spacing in ln IML around theta_ida:")
print(k_df["dln_ims_at_theta_ida"].describe().round(4).to_string(), "\n")

# Exact PCHIP slope at theta_ida: the analytic first derivative of the cubic.
k_exact_ida = np.array([-float(k_interp[s](np.log(t), 1))
                        for (s, _), t in k_df["theta_ida"].items()])

delta_rows = []
for d in (0.001, 0.005, 0.01, 0.05, 0.1, 0.18, 0.36):    # 0.18 ~ half, 0.36 ~ one grid step
    k_d = np.array([point_slope(k_interp[s], t, d) for (s, _), t in k_df["theta_ida"].items()])
    err = np.abs(k_d - k_exact_ida)
    delta_rows.append({"delta": d, "median_k": np.nanmedian(k_d),
                       "median_abs_err": np.nanmedian(err), "max_abs_err": np.nanmax(err)})
k_delta_sensitivity = pd.DataFrame(delta_rows).set_index("delta")

# The delta used in the table must reproduce k_local_ida.
assert np.allclose([point_slope(k_interp[s], t, K_DELTA) for (s, _), t in k_df["theta_ida"].items()],
                   k_df["k_local_ida"], equal_nan=True)

print("k_local_ida against the exact PCHIP derivative, by delta:")
print(k_delta_sensitivity.to_string(float_format=lambda v: f"{v:.3g}"))

In [ ]:
# Summary of the hazard slopes.
k_est_cols = [f"{e}_{arm}" for arm in K_ARMS for e in ("k_local", "k_window", "k_fosm")]

# Distribution of every k column, overall and per storey.
print("All 120 rows:")
print(k_df[k_est_cols].describe().round(3).to_string(), "\n")
for n_storeys, grp in k_df.groupby(level="n_storeys"):
    print(f"{n_storeys}-storey ({len(grp)} rows):")
    print(grp[k_est_cols].describe().round(3).to_string(), "\n")

# Median k per arm, overall and per storey: one compact table.
k_median = pd.concat(
    {"all": k_df[k_est_cols].median(),
     **{f"{n}s": g[k_est_cols].median() for n, g in k_df.groupby(level="n_storeys")}},
    axis=1)
print("Median k:")
print(k_median.round(3).to_string(), "\n")

# k_local (the curve at theta) against k_fosm (what the risk integral responds to).
# They are equal under a pure power-law hazard; the difference is curvature.
k_agree = pd.DataFrame({
    arm: {"corr": k_df[f"k_local_{arm}"].corr(k_df[f"k_fosm_{arm}"]),
          "median(local - fosm)": (k_df[f"k_local_{arm}"] - k_df[f"k_fosm_{arm}"]).median(),
          "max|local - fosm|": (k_df[f"k_local_{arm}"] - k_df[f"k_fosm_{arm}"]).abs().max()}
    for arm in K_ARMS}).T
print("k_local against k_fosm:")
print(k_agree.round(3).to_string(), "\n")

# d ln(lambda)/d ln(beta): the closed-form k^2 beta^2 against the numerical value.
k_beta_slope = pd.DataFrame({
    arm: {"median kbeta2": k_df[f"kbeta2_{arm}"].median(),
          "median s_beta_fosm": k_df[f"s_beta_fosm_{arm}"].median()}
    for arm in K_ARMS}).T
print("d ln(lambda) / d ln(beta), closed form against numerical:")
print(k_beta_slope.round(3).to_string(), "\n")

# Rows with at least one NaN value in each arm's columns.
k_nan_rows = pd.Series({arm: int(k_df.filter(regex=f"_{arm}$").isna().any(axis=1).sum())
                        for arm in K_ARMS}, name="rows with NaN")
print(k_nan_rows.to_string())

# Scatter of k_local_ida against k_fosm_ida, one colour per storey, with a 1:1 line.
# Points below the line are rows where the risk integral responds less steeply
# than the hazard curve's slope at theta.
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for n_storeys, colour in ((3, "#2a78d6"), (5, "#eb6834")):
    grp = k_df.xs(n_storeys, level="n_storeys")
    ax.scatter(grp["k_local_ida"], grp["k_fosm_ida"], s=22, color=colour,
               edgecolor="w", linewidth=0.6, label=f"{n_storeys}-storey", zorder=3)
k_lims = [np.nanmin(k_df[["k_local_ida", "k_fosm_ida"]].to_numpy()) - 0.2,
          np.nanmax(k_df[["k_local_ida", "k_fosm_ida"]].to_numpy()) + 0.2]
ax.plot(k_lims, k_lims, color="0.5", linewidth=1, linestyle="--", label="1:1", zorder=2)
ax.set_xlim(k_lims)
ax.set_ylim(k_lims)
ax.set_aspect("equal")
ax.set_xlabel(r"$k_{local}$ at $\theta_{IDA}$")
ax.set_ylabel(r"$k_{fosm}$ at $\theta_{IDA}$")
ax.grid(color="0.9", linewidth=0.6, zorder=0)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FC_DATA_DIR / "hazard_slope_k_local_vs_fosm.png", dpi=300, facecolor="w")
plt.show()

In [18]:
# Produce and save every assessment figure into FC_DATA_DIR (38 PNGs; the file
# names are listed in the docstring of fca.plot_assessment_figures). Figures are
# closed after saving; pass plot_show=True to keep them open in the notebook.
fca.plot_assessment_figures(assessment_df, FC_DATA_DIR, dpi=300, plot_show=False)

c:\Users\clemettn\Documents\phd\.venv\Lib\site-packages\matplotlib\scale.py:170: RuntimeWarning: divide by zero encountered in log
  return self._forward(values)
c:\Users\clemettn\Documents\phd\.venv\Lib\site-packages\matplotlib\scale.py:170: RuntimeWarning: divide by zero encountered in log
  return self._forward(values)
c:\Users\clemettn\Documents\phd\.venv\Lib\site-packages\matplotlib\scale.py:170: RuntimeWarning: divide by zero encountered in log
  return self._forward(values)


In [19]:
# MAFC of every site/structure row on a log axis: one figure with MSA-SS (true),
# IDA-FX and the corrected MSA-SS estimate, and one with MSA-SS and IDA-FX only.
fca.plot_mafe_by_site(assessment_df, include_est=True,
                      fp=FC_DATA_DIR / "mafe_by_site_all.png", dpi=300, plot_show=False)
fca.plot_mafe_by_site(assessment_df, include_est=False,
                      fp=FC_DATA_DIR / "mafe_by_site_ss_vs_ida.png", dpi=300, plot_show=False)

(<Figure size 936x288 with 1 Axes>, <Axes: ylabel='$\\lambda_C$ (1/yr)'>)

In [ ]:
# MAFC with 95% CI bars, one figure per storey group. At each site IDA-FX,
# MSA-SS (true) and the estimate are offset sideways so the bars do not overlap.
for n_storeys in (3, 5):
    fca.plot_mafe_ci_by_site(assessment_df, n_storeys, fp=FC_DATA_DIR / f"mafe_ci_{n_storeys}s.png",
                             dpi=300, plot_show=False)